**[Source]** Donghwan Project (모델 비교 흐름: 후보 모델을 같은 조건에서 비교하고 Validation으로 선택) + Jisoo Project 8번(subset 기준선 결과)
**[Status]** ADAPTED
**[Role]** KoBART vs pko-T5(vs ET5) 모델 비교. 동환의 평가 지표(Balanced EM 등)를 지수 subset 예측에 적용하고, 07번에서 사전 등록한 규칙으로 선택 여부를 판정
**[Modification]** 동환의 3,000/300행 비교 대신 지수의 고정 subset(Train 100,000/Validation 5,000)을 사용. Part A는 지수가 이미 만든 예측을 재사용(07번에서 subset 해시 확인). Part B(ET5)는 GPU·가중치가 필요해 기본 꺼짐.
**Test는 열지 않는다. subset screening 결과는 최종 성능이 아니다.**

# 08. 모델 비교 (Validation subset)

- 비교 조건(고정): 같은 Train subset 100,000행 · 같은 Validation subset 5,000행 · seed 42 · 1 epoch · lr 5e-5 · 유효 batch 16 · Greedy.
- **Part A (CPU, 실제 실행)**: 지수 8번 예측(KoBART, pko-T5)에 동환식 지표를 적용하고 문서 단위 bootstrap으로 차이를 검정.
- **Part B (GPU 필요, 기본 꺼짐)**: ET5 후보. 가중치가 없으면 실행하지 않고 “미실행”으로 기록한다. 실행하지 않은 결과를 만들지 않는다.
- 판정 규칙은 07번에서 미리 등록한 것을 그대로 쓴다(결과를 본 뒤 바꾸지 않음).

In [1]:
# [공통 준비] 경로 · 재현성 · 05번 검증 통과 확인
import os, sys, json, re, time, math, random, hashlib, platform
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 70); pd.set_option("display.unicode.east_asian_width", True)

def _find_root():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "config" / "paths.json").exists():
            return c
    raise FileNotFoundError("config/paths.json이 있는 통합 프로젝트 루트를 찾지 못했습니다(notebooks 폴더에서 실행하세요).")
ROOT = _find_root(); sys.path.insert(0, str(ROOT / "src"))
import common, ko_metrics as km
P = common.load_paths(ROOT)
SEED = 42; random.seed(SEED); np.random.seed(SEED)

VER = json.loads((P.PROCESSED / "verification_05" / "verification_result.json").read_text(encoding="utf-8"))
assert VER["verdict"] == "PASS", "05번 전처리 검증이 PASS가 아닙니다 → 모델링을 진행하지 않습니다"
MAN = common.manifest(P)
assert VER["sha256_actual"]["train"] == MAN["sha256"]["train.jsonl"], "검증 이후 데이터가 바뀌었습니다(05번을 다시 실행하세요)"
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__, "|", platform.platform())
print("통합 프로젝트:", ROOT); print("지수 최종 데이터:", P.DATA_DIR, "|", MAN["dataset_version"])
print("05번 검증:", VER["verdict"], "@", VER["verified_at"], "| metric backends:", km.BACKENDS)

Python 3.10.12 | pandas 2.3.3 | numpy 2.2.6 | Linux-6.8.0-138-generic-x86_64-with-glibc2.35
통합 프로젝트: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/integrated_korean_correction
지수 최종 데이터: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/data/preprocessed_final | preprocessed_final_v1
05번 검증: PASS @ 2026-09-21 05:30:57 | metric backends: {'rapidfuzz': False, 'sacrebleu': False}


In [2]:
# [셀 1] Validation과 지수 subset 예측 로드 + 같은 행인지 검증
VAL = common.read_split(P, "validation", columns=["document_id", "utterance_id", "input", "target"])
VBY = {r["utterance_id"]: r for r in VAL}
SUB = P.J_OUT / "subset_baseline"
vs = json.loads((SUB / "val_subset_ids.json").read_text(encoding="utf-8"))
ids = vs["utterance_ids"]; assert len(ids) == len(set(ids)) == vs["n"] == 5000 and all(i in VBY for i in ids)
rows = [VBY[i] for i in ids]
SRC, TGT, DOC = [r["input"] for r in rows], [r["target"] for r in rows], [r["document_id"] for r in rows]
PRED = {"input-copy": list(SRC)}
for m in ("KoBART", "pko-T5"):
    d = pd.read_csv(SUB / f"val_predictions_{m}.csv", encoding="utf-8-sig", keep_default_na=False)
    assert d.utterance_id.tolist() == ids or set(d.utterance_id) == set(ids), f"{m}: 예측 id가 Validation subset과 다름"
    d = d.set_index("utterance_id").loc[ids]
    assert d["input"].tolist() == SRC and d["target"].tolist() == TGT, f"{m}: input/target이 현재 Validation과 다름"
    PRED[m] = d["prediction"].astype(str).tolist()
    print(f"{m}: {len(d):,}행 검증 통과 | 빈 출력 {sum(1 for x in PRED[m] if not x.strip())}행")
print("문서 수(subset):", len(set(DOC)))
print("교정 필요 행:", sum(s != t for s, t in zip(SRC, TGT)), "/ 원문 유지 행:", sum(s == t for s, t in zip(SRC, TGT)))

KoBART: 5,000행 검증 통과 | 빈 출력 14행
pko-T5: 5,000행 검증 통과 | 빈 출력 0행
문서 수(subset): 1466
교정 필요 행: 4298 / 원문 유지 행: 702


In [3]:
# [셀 2] 지표 계산 — 지수 ROUGE(원문 기준)를 재현할 수 있는지 먼저 확인한 뒤 동환식 지표를 추가
REC = json.loads((SUB / "subset_baseline_results.json").read_text(encoding="utf-8"))["rouge_mean"]
ROW = {m: km.score_rows(PRED[m], TGT) for m in PRED}
chk = []
for m in PRED:
    mine, theirs = float(ROW[m]["R2_어절"].mean()), REC[m]["R2_어절"]
    chk.append({"모델": m, "이번 계산 R2_어절": round(mine, 4), "지수 기록": theirs, "차이": round(mine - theirs, 4), "재현": abs(mine - theirs) < 1e-3})
CHK = pd.DataFrame(chk); print(CHK.to_string(index=False)); assert CHK["재현"].all(), "지수 ROUGE를 재현하지 못함 → 정의 불일치, 이후 비교 중단"
MET = {}
for m in PRED:
    for nf in (True, False):
        MET[(m, "NFKC" if nf else "raw")] = km.generation_metrics(SRC, TGT, PRED[m], nfkc=nf)
df = pd.DataFrame(MET).T
cols = ["exact_match", "need_correction_em", "unchanged_em", "balanced_em", "cer", "chrf", "rouge2_donghwan", "over_correction_rate", "miss_rate"]
print(df[cols].round(4).to_string())

      모델  이번 계산 R2_어절  지수 기록  차이  재현
input-copy             0.3921     0.3921  -0.0  True
    KoBART             0.7085     0.7085  -0.0  True
    pko-T5             0.7382     0.7382  -0.0  True
                 exact_match  need_correction_em  unchanged_em  balanced_em     cer    chrf  rouge2_donghwan  over_correction_rate  miss_rate
input-copy NFKC       0.1408              0.0000        1.0000       0.5000  0.1594  0.7908           0.2821                0.0000     1.0000
           raw        0.1404              0.0000        1.0000       0.5000  0.1603  0.7896           0.2817                0.0000     1.0000
KoBART     NFKC       0.5500              0.5116        0.7841       0.6479  0.0617  0.8939           0.7025                0.2159     0.0365
           raw        0.4316              0.4104        0.5613       0.4858  0.1235  0.8049           0.6247                0.4387     0.0340
pko-T5     NFKC       0.6172              0.5824        0.8295       0.7060  0.0491  0.9172   

### 지표 읽는 법 (해석 전 확인)
- **Balanced EM** = (교정 필요 EM + 원문 유지 EM)/2. 입력을 그대로 복사하면 정확히 0.5가 되므로 0.5가 바닥선이다. Exact Match만 보면 “원문을 그대로 두는 모델”이 유리해지는 편향을 막는다.
- **NFKC vs raw**: 지수 8·11단계에서 확인된 tokenizer 왕복 변환(유니코드 정규화 차이) 때문에 raw EM은 실제 교정 실패가 아닌 것도 오답으로 센다. 두 값을 나란히 보고, 어느 쪽으로 결론이 나든 같은 방향인지 확인한다.
- **ROUGE**(어절/문자)는 어휘 겹침 지표라 의미가 같은 다른 표현을 낮게 평가하고, 한 글자 교정 차이에도 어절 R2가 급락한다. 단독 근거로 쓰지 않는다.
- chrF/CER은 이 환경에서 직접 구현한 값이며 sacrebleu와의 일치 여부는 미검증(실행 후 확인).

In [4]:
# [셀 3] 문서 단위 bootstrap 비교(pko-T5 − KoBART) + 07번에서 등록한 선택 규칙 적용
A, B = "pko-T5", "KoBART"
need = np.array([s != t for s, t in zip(SRC, TGT)])
res = {}
for name, key in (("전체 어절 R2 (주 지표)", None), ("교정 필요 행 어절 R2", "need")):
    a, b = ROW[A]["R2_어절"], ROW[B]["R2_어절"]
    if key == "need": a, b, docs = a[need], b[need], np.array(DOC)[need]
    else: docs = np.array(DOC)
    res[name] = km.boot_paired_diff(a, b, docs, n_boot=2000, seed=42)
def _ex(m): return np.array([common_norm(p) == common_norm(t) for p, t in zip(PRED[m], TGT)])
common_norm = km.normalize_for_evaluation
res["Balanced EM (NFKC)"] = km.boot_balanced_em_diff(_ex(A), _ex(B), need, DOC, n_boot=2000, seed=42)
T = pd.DataFrame([{"비교": k, "차이(pko-T5−KoBART)": round(v[0], 4), "CI95 하한": round(v[1], 4), "CI95 상한": round(v[2], 4), "CI가 0 제외": (v[1] > 0 or v[2] < 0)} for k, v in res.items()])
print(T.to_string(index=False))

# 사전 등록 규칙
elig = {}
for m in (A, B):
    elig[m] = {"교정 필요 행 R2 > 입력 복사": bool(ROW[m]["R2_어절"][need].mean() > ROW["input-copy"]["R2_어절"][need].mean()),
               "Balanced EM(NFKC) > 0.5": bool(MET[(m, "NFKC")]["balanced_em"] > 0.5)}
print("\n적격성:", json.dumps(elig, ensure_ascii=False))
main = res["전체 어절 R2 (주 지표)"]
winner = A if main[1] > 0 else (B if main[2] < 0 else None)
bal = res["Balanced EM (NFKC)"]
conflict = (winner == A and bal[0] < 0) or (winner == B and bal[0] > 0)
print("주 지표 승자:", winner, "| Balanced EM 점추정과 충돌:", conflict)
SEL = {"winner_primary": winner, "eligibility": elig, "balanced_em_conflict": bool(conflict), "bootstrap": {k: list(map(float, v)) for k, v in res.items()},
       "agrees_with_jisoo_step8_selection(pko-T5)": winner == "pko-T5", "scope": "subset(100k/5k) screening, Test 미사용, 최종 성능 아님"}

                  비교  차이(pko-T5−KoBART)  CI95 하한  CI95 상한  CI가 0 제외
전체 어절 R2 (주 지표)               0.0297     0.0221     0.0374         True
  교정 필요 행 어절 R2               0.0332     0.0246     0.0422         True
    Balanced EM (NFKC)               0.0570     0.0426     0.0712         True

적격성: {"pko-T5": {"교정 필요 행 R2 > 입력 복사": true, "Balanced EM(NFKC) > 0.5": true}, "KoBART": {"교정 필요 행 R2 > 입력 복사": true, "Balanced EM(NFKC) > 0.5": true}}
주 지표 승자: pko-T5 | Balanced EM 점추정과 충돌: False


In [5]:
# [셀 4] 비용(시간·메모리)과 그림 — 지수 8번에서 실제 측정된 값
R8 = json.loads((SUB / "subset_baseline_results.json").read_text(encoding="utf-8"))
cost = pd.DataFrame([{"모델": m, "학습 시간(초)": R8["train_results"][m]["train_time_sec"], "최대 GPU 메모리(GB)": R8["train_results"][m]["peak_gpu_mem_gb"],
                      "Validation 5,000행 생성(초)": R8["inference"][m]["sec_total"], "1,000행당 생성(초)": R8["inference"][m]["sec_per_1000"]} for m in (B, A)])
print(cost.to_string(index=False)); print("※ 같은 GPU·같은 세션에서 측정했다고 기록된 값(지수 8번). 재측정하지 않음.")
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
names = ["input-copy", B, A]; vals = [float(ROW[m]["R2_어절"].mean()) for m in names]
ax[0].bar(names, vals, color=["#bbb", "#6a9fd4", "#e08a4a"]); ax[0].set_ylabel("Word-level ROUGE-2 F1"); ax[0].set_title("Validation subset (n=5,000)")
for i, v in enumerate(vals): ax[0].text(i, v + 0.01, f"{v:.3f}", ha="center")
bv = [MET[(m, "NFKC")]["balanced_em"] for m in names]
ax[1].bar(names, bv, color=["#bbb", "#6a9fd4", "#e08a4a"]); ax[1].axhline(0.5, ls="--", c="k", lw=0.8); ax[1].set_ylabel("Balanced EM (NFKC)"); ax[1].set_title("0.5 = copy baseline")
for i, v in enumerate(bv): ax[1].text(i, v + 0.01, f"{v:.3f}", ha="center")
plt.tight_layout(); (P.REPORTS).mkdir(exist_ok=True, parents=True); plt.savefig(P.REPORTS / "fig08_model_comparison.png", dpi=150); plt.close()
(P.RUNS / "model_comparison_08.json").write_text(json.dumps({"selection": SEL, "metrics": {f"{k[0]}|{k[1]}": v for k, v in MET.items()}, "cost": cost.to_dict("records"), "et5_status": "see Part B"}, ensure_ascii=False, indent=2, default=float), encoding="utf-8")
print("저장: runs/model_comparison_08.json, reports/fig08_model_comparison.png")

  모델  학습 시간(초)  최대 GPU 메모리(GB)  Validation 5,000행 생성(초)  1,000행당 생성(초)
KoBART         1379.7                 2.55                         31.5                6.30
pko-T5         3353.3                 6.21                         62.9               12.58
※ 같은 GPU·같은 세션에서 측정했다고 기록된 값(지수 8번). 재측정하지 않음.
저장: runs/model_comparison_08.json, reports/fig08_model_comparison.png


## 해석 (Part A)
(실행 후 작성)

# Part B. ET5 후보 (GPU 필요 · 기본 꺼짐)
ET5 사전학습 가중치가 필요하다(`config/paths.json`의 `et5_model_dir`). 아래 셀은 **`RUN_ET5=True`로 바꾸고 GPU가 있을 때만** 실행된다. 지수 subset(100,000/5,000)을 그대로 쓰고, 입력 앞에 `맞춤법 교정: `을 붙인다(동환 방식). max_length는 지수 5번 규칙(Train 잘림 ≤ 0.1%)으로 결정한다. 학습률은 07번에서 등록한 격자 {3e-5, 5e-5}를 앞 10,000행/2,000행에서 비교한다.
**이 셀들은 작성 환경(GPU·ET5 없음)에서 실행해 보지 못했다 — 실행 후 확인.**

In [6]:
# [셀 5] ET5 screening (guarded)
RUN_ET5 = False
ET5_DIR = P.ET5_DIR if getattr(P, "ET5_DIR", None) else None
ET5_STATUS = {"run": False, "reason": None}
try:
    import torch; HAS_GPU = torch.cuda.is_available()
except Exception:
    HAS_GPU = False
if not RUN_ET5:
    ET5_STATUS["reason"] = "RUN_ET5=False (기본값)"
elif not HAS_GPU or not ET5_DIR or not Path(ET5_DIR).exists():
    ET5_STATUS["reason"] = f"GPU={HAS_GPU}, ET5_DIR={ET5_DIR} → 조건 미충족"
else:
    import seq2seq_tools as S2S
    from transformers import AutoTokenizer
    PREFIX = "맞춤법 교정: "
    tok = AutoTokenizer.from_pretrained(ET5_DIR)
    TR = {r["utterance_id"]: r for r in common.read_split(P, "train", columns=["utterance_id", "input", "target"])}
    tids = json.loads((SUB / "train_subset_ids.json").read_text(encoding="utf-8"))["utterance_ids"]
    trs = [TR[i] for i in tids]
    max_in, r_in = S2S.choose_max_length(tok, [r["input"] for r in trs], prefix=PREFIX)
    max_tg, r_tg = S2S.choose_max_length(tok, [r["target"] for r in trs], target=True)
    print("ET5 max_length(in/out):", max_in, max_tg)
    ex, info = S2S.build_examples(tok, [r["input"] for r in trs], [r["target"] for r in trs], max_in, max_tg, prefix=PREFIX); print(info)
    vex, _ = S2S.build_examples(tok, SRC[:2000], TGT[:2000], max_in, max_tg, prefix=PREFIX)
    hpo = {}
    for lr in json.loads((P.CONFIG / "experiment_config.json").read_text(encoding="utf-8"))["hpo_ET5"]["lr_grid"]:
        model, s = S2S.train(ET5_DIR, tok, ex[:10000], P.RUNS / f"et5_hpo_lr{lr}", lr=lr, micro=8, accum=2, seed=SEED)
        hpo[lr] = S2S.eval_loss(model, tok, vex); print("lr", lr, "val loss", hpo[lr]); del model
    lr_best = 5e-5 if abs(hpo[3e-5] - hpo[5e-5]) / hpo[5e-5] < 0.01 else min(hpo, key=hpo.get)
    model, s = S2S.train(ET5_DIR, tok, ex, P.RUNS / "et5_screening", lr=lr_best, micro=8, accum=2, seed=SEED)
    pred = S2S.generate_resumable(model, tok, SRC, P.RUNS / "et5_screening" / "pred_part.jsonl", max_in, max_tg, {"num_beams": 1, "do_sample": False}, prefix=PREFIX)
    PRED["ET5"] = pred; ET5_STATUS = {"run": True, "lr_best": lr_best, "hpo_val_loss": hpo, "train_summary": s, "max_length": [max_in, max_tg]}
    m5 = km.generation_metrics(SRC, TGT, pred); print(pd.Series(m5).round(4).to_string())
    ET5_STATUS["metrics_nfkc"] = m5
p = P.RUNS / "model_comparison_08.json"; d = json.loads(p.read_text(encoding="utf-8")); d["et5_status"] = ET5_STATUS
p.write_text(json.dumps(d, ensure_ascii=False, indent=2, default=float), encoding="utf-8"); print("ET5 상태:", ET5_STATUS.get("run"), ET5_STATUS.get("reason"))

ET5 상태: False RUN_ET5=False (기본값)


## 해석 (Part B)
(실행 후 작성)